In [3]:
import pygame
import random
import sys

pygame.init()

# Screen
WIDTH, HEIGHT = 400, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Car Racing Game")

clock = pygame.time.Clock()

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GRAY = (40, 40, 40)
YELLOW = (255, 255, 0)
RED = (220, 0, 0)
GREEN = (0, 200, 0)

# Fonts
font_big = pygame.font.SysFont("Arial", 35, bold=True)
font_small = pygame.font.SysFont("Arial", 20)

# Road animation
road_y = 0

def draw_road():
    global road_y
    screen.fill((20, 20, 20))

    # road base
    pygame.draw.rect(screen, GRAY, (50, 0, 300, HEIGHT))

    # moving lane lines
    for i in range(0, HEIGHT, 40):
        pygame.draw.rect(screen, WHITE, (195, i + road_y, 10, 20))

    road_y += 5
    if road_y >= 40:
        road_y = 0


# Player
class Player:
    def __init__(self):
        self.x = 175
        self.y = 480
        self.speed = 6

    def move(self, keys):
        if keys[pygame.K_LEFT] and self.x > 60:
            self.x -= self.speed
        if keys[pygame.K_RIGHT] and self.x < 290:
            self.x += self.speed

    def draw(self):
        # car body
        pygame.draw.rect(screen, GREEN, (self.x, self.y, 50, 100), border_radius=10)
        pygame.draw.rect(screen, BLACK, (self.x + 5, self.y + 10, 40, 30))  # windshield


# Enemy
class Enemy:
    def __init__(self):
        self.reset()

    def reset(self):
        self.x = random.randint(60, 290)
        self.y = random.randint(-600, -100)
        self.speed = random.randint(4, 8)

    def move(self):
        self.y += self.speed
        if self.y > HEIGHT:
            self.reset()

    def draw(self):
        pygame.draw.rect(screen, RED, (self.x, self.y, 50, 100), border_radius=10)
        pygame.draw.rect(screen, BLACK, (self.x + 5, self.y + 10, 40, 30))


# Collision
def is_collision(p, e):
    return pygame.Rect(p.x, p.y, 50, 100).colliderect(
        pygame.Rect(e.x, e.y, 50, 100)
    )


# Text
def draw_text(text, font, color, x, y, center=True):
    render = font.render(text, True, color)
    rect = render.get_rect()
    if center:
        rect.center = (x, y)
    else:
        rect.topleft = (x, y)
    screen.blit(render, rect)


# Menu
def show_menu():
    while True:
        screen.fill(BLACK)
        draw_text("CAR RACING", font_big, WHITE, WIDTH // 2, 200)
        draw_text("Press ENTER to Start", font_small, WHITE, WIDTH // 2, 300)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            if event.type == pygame.KEYDOWN and event.key == pygame.K_RETURN:
                return

        pygame.display.update()


# Game Over
def game_over(score):
    while True:
        screen.fill(BLACK)
        draw_text("GAME OVER", font_big, RED, WIDTH // 2, 200)
        draw_text(f"Score: {score}", font_small, WHITE, WIDTH // 2, 280)
        draw_text("Press R to Restart", font_small, WHITE, WIDTH // 2, 350)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_r:
                    return

        pygame.display.update()


# Game Loop
def game_loop():
    player = Player()
    enemies = [Enemy() for _ in range(3)]
    score = 0

    while True:
        draw_road()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()

        keys = pygame.key.get_pressed()
        player.move(keys)

        for enemy in enemies:
            enemy.move()
            enemy.draw()

            if is_collision(player, enemy):
                # crash flash effect
                screen.fill(RED)
                pygame.display.update()
                pygame.time.delay(100)
                return score

        player.draw()

        score += 1
        draw_text(f"Score: {score}", font_small, WHITE, 10, 10, center=False)

        pygame.display.update()
        clock.tick(60)


# Run
while True:
    show_menu()
    final_score = game_loop()
    game_over(final_score)

SystemExit: 